# Tier 2 — Delhi AQI forecasting experiments (Colab)

Runs every Tier 2 task of `docs/IMPROVEMENT_PLAN.md` end to end, on Colab instead of a laptop.
Every number is produced by a script in `src/` (this notebook only calls them and displays
their output files), and every display cell checks that the file was **written in this run**,
so stale results from an earlier run can never be mistaken for fresh ones.

| Section | Task | Script | Output |
| :--- | :--- | :--- | :--- |
| 3 | T2.1 recompute CPCB AQI check | `src/features/compute_cpcb_aqi.py` | `reports/features/aqi_recompute_validation.md` |
| 4 | T2.1 24h model on 2024 + 2025 | `src/models/retrain_24h_extended.py` | `reports/modeling/24h/final/extended_training_metrics.json` |
| 5 | T2.2 rolling-origin backtest | `src/evaluation/rolling_origin_backtest.py` | `reports/modeling/24h/rolling_origin_backtest.{csv,md}` |
| 6 | T2.4 weather ablation (+ optional fires) | `src/features/exogenous_drivers.py`, `src/models/ablation_exogenous.py` | `reports/modeling/exogenous/ablation.{csv,md}` |
| 7 | T2.3 raw PM2.5 forecasts | `src/models/train_pm25_forecast.py` | `reports/modeling/pm25/` |
| 8 | T2.5 quantile + conformal + P(AQI ≥ 401) | `src/models/train_quantile_forecast.py` | `reports/modeling/probabilistic/` |
| 9 | Headline table | `src/evaluation/build_metrics_table.py` | `reports/metrics/headline_metrics.{csv,md}` |

**Rules the scripts follow:** hyper-parameters, calendar features and target form are chosen on
validation (Sep–Oct 2025) only; test (Nov–Dec 2025) is scored once; training rows in the 24 h
before a window are dropped (embargo); persistence and skill (`1 − model / persistence`) are
always shown. Only scikit-learn is needed (HistGradientBoosting), no LightGBM.

### Compute profile
`PROFILE = "max"` (set it in the toggles cell after a short `"smoke"` run, which only checks that every section runs) spends more compute to get the best *validation-selected* models:
a random hyper-parameter search for the PM2.5, quantile and weather-ablation gradient-boosting models, boosting
iteration counts chosen on validation, and 10,000 bootstrap resamples. It writes to `<results folder>/max/`
and never overwrites the accepted `standard` results. Two honest caveats: the validation window (Sep-Oct) is much
calmer than the winter test window, so more tuning can improve validation without improving test (the test
numbers are the verdict, scored once), and the 24h AQI point forecast is limited by data, not compute.

### How to run
1. Put the project folder in Google Drive. It must contain `src/`, `models/24h/`, `tests/` and either
   `data/processed/features_extended.parquet` **or** `Og Data/` (the notebook then rebuilds the features).
2. Edit `PROJECT_DIR` in the first code cell and the toggles in the second.
3. `Runtime → Run all`. Section 7 (PM2.5, 12 model fits) and section 8 (quantile models) are the slow ones;
   switch them off with the toggles to run the rest quickly. Results are written back into the Drive
   folder, so a disconnected runtime does not lose finished sections.

## 1 — Setup

In [ ]:
# --- Locate the project (Google Drive) and define helpers --------------------
import os, pathlib, subprocess, sys, time

PROJECT_DIR = "/content/drive/MyDrive/projects/Air quality risk forecasting"  # <-- EDIT ME

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped (not on Colab?):", exc)

if not pathlib.Path(PROJECT_DIR, "src").is_dir():
    raise FileNotFoundError(
        f"{PROJECT_DIR!r} has no src/ folder. Fix PROJECT_DIR (the folder that contains src/, models/, tests/)."
    )
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("CWD:", os.getcwd())

RUN_START = time.time()


def run(*args, check=True):
    # Run `python <args>` from the project root, stream its output, and fail loudly.
    # (Unlike a bare `!python`, a failing script stops the notebook here.)
    t0 = time.time()
    print("$ python", " ".join(args), flush=True)
    proc = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, cwd=PROJECT_DIR)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    print(f"[exit {code} after {time.time() - t0:.0f}s]")
    if check and code != 0:
        raise RuntimeError(f"`python {' '.join(args)}` failed with exit code {code}; see the output above.")
    return code


def fresh(path):
    # True if `path` exists and was written during this notebook run.
    p = pathlib.Path(path)
    return p.exists() and p.stat().st_mtime >= RUN_START - 1


def require_fresh(path, ran):
    # Guard for display cells: only show results written in THIS run (never stale numbers).
    # Set SHOW_STALE = True in the toggles cell to inspect an older file anyway.
    p = pathlib.Path(path)
    if not p.exists():
        print(f"!! {path} does not exist yet - run the section above.")
        return False
    if fresh(p):
        return True
    why = "this section was switched off" if not ran else "the script did not rewrite it"
    if SHOW_STALE:
        print(f"!! {path} is from an EARLIER run ({why}); the numbers below are STALE.")
        return True
    print(f"-- skipped {path}: not written in this run ({why}). Switch the section on, or set SHOW_STALE = True.")
    return False


# --- Frozen 24h predictions must never change (two Colab runs came back with this file replaced by the
# extended-model predictions, which silently corrupts the headline table). Verify by hash; restore from
# the backup shipped in the bundle (data/frozen/) if needed; refuse to continue if it cannot be restored.
import hashlib, shutil
FROZEN = pathlib.Path("reports/modeling/24h/final/final_predictions.parquet")
FROZEN_BACKUP = pathlib.Path("data/frozen/final_predictions.parquet")
FROZEN_MD5 = "a7e9e26fc5747bab0b2ac728fa6a9870"


def _md5(path):
    return hashlib.md5(pathlib.Path(path).read_bytes()).hexdigest()


def ensure_frozen():
    if FROZEN.exists() and _md5(FROZEN) == FROZEN_MD5:
        print("frozen 24h predictions: OK")
        return
    if FROZEN_BACKUP.exists() and _md5(FROZEN_BACKUP) == FROZEN_MD5:
        FROZEN.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(FROZEN_BACKUP, FROZEN)
        print("!! frozen 24h predictions had been overwritten; restored from", FROZEN_BACKUP)
        return
    raise RuntimeError(f"{FROZEN} is not the frozen file and there is no valid backup at {FROZEN_BACKUP}. "
                       "Unzip tier_2_patch_v3.zip into the project folder and run this cell again.")


In [ ]:
# --- Compute profile ----------------------------------------------------------
# Recommended order:  1) "smoke"  -> ~5 minutes, tiny models, only checks that every section runs end to end
#                     2) "max"    -> the real run (results in <folder>/max/)
# "max"      random hyper-parameter search for the PM2.5, quantile and weather-ablation GBMs (the default
#            config is always one candidate), iteration counts chosen on validation, 10,000 bootstrap
#            resamples. Rough run time ~1-4 h of CPU for sections 6-8 together (NOT measured; depends on the
#            Colab CPU; more cores help, a GPU does not). Results never overwrite the accepted "standard" ones.
# "standard" the fixed configuration behind the accepted results (~20-40 min).
# Tip: run sections 6, 7, 8 as separate sessions (turn the other RUN_* toggles off) so a disconnect only
# costs one section; every finished section is already saved in the Drive folder.
PROFILE = "smoke"        # "smoke", "max" or "standard"
assert PROFILE in ("smoke", "max", "standard"), PROFILE
os.environ["T2_PROFILE"] = PROFILE

def out(base):
    # results folder for the chosen profile (standard -> base, max -> base/max)
    return pathlib.Path(base) / ("" if PROFILE == "standard" else PROFILE)


def rp(base, name):
    return str(out(base) / name)


print("profile:", PROFILE, "| results under", out("reports/modeling/pm25"))
if PROFILE == "max" and not (out("reports/modeling/pm25").parent / "smoke" / "variant_selection.csv").exists():
    print("!! WARNING: no smoke run found. Run PROFILE = 'smoke' first (a few minutes) so a bug in the search "
          "code does not cost you hours.")

# --- Toggles ----------------------------------------------------------------
RUN_RECOMPUTE_CHECK = True   # ~30 s   T2.1 validate the recomputed AQI against official 2025 AQI
RUN_EXTENDED_24H    = True   # ~1 min  T2.1 24h Ridge hybrid trained on 2024 + 2025
RUN_ROLLING_ORIGIN  = True   # ~1 min  T2.2 winter-2024 / winter-2025 folds
RUN_WEATHER_ABLATION = True  # network + a few minutes  T2.4 (needs internet for Open-Meteo)
RUN_FIRES           = False  # needs a free NASA FIRMS MAP_KEY (see section 6)
RUN_PM25            = True   # slowest: 12 gradient-boosting fits  T2.3
RUN_QUANTILE        = True   # slow: quantile + classifier models  T2.5
REBUILD_HEADLINE    = True   # seconds  regenerate reports/metrics/headline_metrics.*
SHOW_STALE          = False  # True: also display results left over from an earlier run (clearly flagged)

## 2 — Preflight: dependencies, data, unit tests

Checks the environment, rebuilds `features_extended.parquet` from the raw CSVs when it is missing
(the 2024 AQI target is **recomputed** from raw telemetry with the CPCB method; 2025 uses official AQI),
and runs the fast unit tests for the split/embargo logic and the AQI recompute.

In [ ]:
import importlib

import numpy as np, pandas as pd, sklearn
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
      "| sklearn", sklearn.__version__)
major, minor = (int(x) for x in sklearn.__version__.split(".")[:2])
assert (major, minor) >= (1, 4), "scikit-learn >= 1.4 is required (native NaN support in HistGradientBoosting)"

missing = [m for m in ("pyarrow", "matplotlib", "joblib", "pytest") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("installed:", missing)

from pathlib import Path
FEATURES = Path("data/processed/features_extended.parquet")
RECOMPUTED = Path("data/processed/recomputed_aqi.parquet")
NEEDED = ["models/24h/feature_names_ab.joblib", "src/models/splits.py", "src/models/phase_6c_refinement.py"]
for rel in NEEDED:
    assert Path(rel).exists(), f"missing {rel} - copy the whole project folder to Drive"

# Guard against an OLD copy of the project on Drive (an earlier run silently used pre-audit scripts
# and even overwrote the frozen 24h predictions). Each marker only exists in the corrected scripts.
MARKERS = {
    "src/features/exogenous_drivers.py": "_get_with_retry",
    "src/models/tuning.py": "search_configs",
    "src/models/train_pm25_forecast.py": "get_profile",
    "src/evaluation/rolling_origin_backtest.py": "purge_train_tail",
    "src/models/train_quantile_forecast.py": "conformal_pad",
    "src/models/ablation_exogenous.py": "blh_window",
    "src/models/retrain_24h_extended.py": "training_rows",
}
stale = [f for f, m in MARKERS.items() if m not in Path(f).read_text()]
assert not stale, f"OLD code on Drive, unzip the latest tier_2_patch_v5.zip over the project folder: {stale}"
print("code version check: OK (corrected scripts)")
ensure_frozen()

if not FEATURES.exists():
    raw = Path("Og Data")
    if not raw.is_dir():
        raise FileNotFoundError(
            "data/processed/features_extended.parquet is missing and 'Og Data/' is not in the project folder, "
            "so the features cannot be rebuilt. Upload one of them to Drive.")
    if not RECOMPUTED.exists():
        run("src/features/compute_cpcb_aqi.py", "--years", "2023", "2024", "2025")
    run("-c", "from src.features.build_features import build_feature_matrix; "
              "build_feature_matrix(source='offline', output_path='data/processed/features_extended.parquet')")
print(FEATURES, "->", round(FEATURES.stat().st_size / 1e6, 1), "MB")

# unit tests: split boundaries / 24h embargo and the CPCB sub-index maths
if run("-m", "pytest", "-q", "tests/test_splits.py", "tests/test_cpcb_aqi.py", check=False) != 0:
    print("!! unit tests failed - the numbers below should not be trusted until this is fixed.")

In [ ]:
# --- Data overview, using the same split helper as every script --------------
import matplotlib.pyplot as plt
from src.models.splits import assign_split, training_rows, VAL_START, EMBARGO_HOURS, model_feature_columns

frame = pd.read_parquet(FEATURES)
frame["split"] = assign_split(frame["timestamp"])
print("shape:", frame.shape)
print("rows per split:", frame["split"].value_counts().to_dict())
print("AQI label source:", frame["aqi_source"].value_counts().to_dict())
print("stations:", frame["station_name"].nunique())

train = training_rows(frame)
kept = len(train); total = int((frame["split"] == "train").sum())
print(f"embargo: {total - kept} rows within {EMBARGO_HOURS}h of the validation start dropped from training")
from src.models.splits import to_kst
assert to_kst(train["timestamp"]).max() < VAL_START - pd.Timedelta(hours=EMBARGO_HOURS) + pd.Timedelta(hours=1), "embargo not applied"
print("features with / without calendar columns:",
      len(model_feature_columns(frame, drop_calendar=False)), "/", len(model_feature_columns(frame, drop_calendar=True)))

monthly = frame.groupby(to_kst(frame["timestamp"]).dt.tz_localize(None).dt.to_period("M"))["aqi_curr"].mean()
monthly.index = monthly.index.astype(str)
ax = monthly.plot(kind="bar", figsize=(11, 3), color="#8c5a2b", title="Mean current AQI by month")
ax.set_xlabel(""); ax.set_ylabel("AQI"); plt.tight_layout(); plt.show()
del frame, train

## 3 — T2.1 · Recomputed AQI vs official AQI

2024 has no official AQI, so it is recomputed from raw telemetry (hourly sub-index max → 24 h rolling
mean, ≥ 16 valid hours). This reproduces the **official 2025** AQI to check the method. The gate
(MAE < 10) passes overall but the error depends on the season (see the by-period table), so 2024 labels
are noisier than 2025 labels.

In [ ]:
from IPython.display import Markdown, display
if RUN_RECOMPUTE_CHECK:
    if Path("Og Data").is_dir():
        run("src/features/compute_cpcb_aqi.py", "--years", "2023", "2024", "--validate")
    else:
        print("'Og Data/' not in the project folder: cannot re-validate; showing the saved report.")
p = "reports/features/aqi_recompute_validation.md"
if require_fresh(p, RUN_RECOMPUTE_CHECK and Path("Og Data").is_dir()):
    display(Markdown(Path(p).read_text()))

## 4 — T2.1 · 24h hybrid trained on 2024 + 2025

The frozen 24h model (50 % persistence + 50 % Ridge, α = 1000, 49 curated features) retrained on
2024-01 → 2025-08 with the 24 h embargo. Compare with the frozen model in section 9.

In [ ]:
import json
if RUN_EXTENDED_24H:
    run("-m", "src.models.retrain_24h_extended")
p = "reports/modeling/24h/final/extended_training_metrics.json"
if require_fresh(p, RUN_EXTENDED_24H):
    print(json.dumps(json.load(open(p)), indent=2))

## 5 — T2.2 · Rolling-origin backtest

Expanding training window, frozen architecture, one fold per winter (2024 scored against recomputed
AQI, 2025 against official AQI), 24 h embargo, 95 % day-block bootstrap CIs on the skill.

In [ ]:
if RUN_ROLLING_ORIGIN:
    run("-m", "src.evaluation.rolling_origin_backtest")
p = rp("reports/modeling/24h", "rolling_origin_backtest.md")
if require_fresh(p, RUN_ROLLING_ORIGIN):
    display(Markdown(Path(p).read_text()))
    ro = pd.read_csv(rp("reports/modeling/24h", "rolling_origin_backtest.csv"))
    fig, ax = plt.subplots(figsize=(5, 3))
    x = np.arange(len(ro)); y = ro["skill_mae"]
    ax.bar(x, y, color="#2a9d8f", yerr=[y - ro["skill_mae_ci_lo"], ro["skill_mae_ci_hi"] - y], capsize=4)
    ax.axhline(0, color="k", lw=0.8); ax.set_xticks(x); ax.set_xticklabels(ro["fold"])
    ax.set_title("24h MAE skill vs persistence (95% CI)"); plt.tight_layout(); plt.show()

## 6 — T2.4 · Exogenous drivers and ablation

Fetches **Open-Meteo ERA5** weather per station (needs internet) and runs the ablation over two panels
(`full_no_blh`: from 2024-01 without boundary-layer height, which the archive lacks before 2024-07-01;
`blh_window`: from 2024-07 with it) and two model families (`ridge_hybrid`, `gbm_residual`).
The `+weather_oracle` group (weather at t+24) is an **upper bound** and is never selected. Selection is
on validation; test is scored once for the selected configuration and its matched baseline.

**Fires (optional):** set `RUN_FIRES = True` and add a free NASA FIRMS key at
<https://firms.modaps.eosdis.nasa.gov/api/area/> as a Colab secret named `FIRMS_MAP_KEY`
(🔑 icon in the left sidebar) or in the prompt below. Without it the fire groups are reported `not_run`.

The fetch is throttled to be kind to the free Open-Meteo API: one station and one year per request with a pause
between them, automatic retries when it answers "too many requests" (HTTP 429), and a disk cache in
`data/processed/exogenous_cache/` so re-running this section (smoke, then max) makes **no new requests**. If an
hourly/daily quota is used up the cell stops with a clear message and keeps what it already fetched; just run it
again later. Set `--refresh` on the command only if you really want to refetch.

In [ ]:
if RUN_WEATHER_ABLATION:
    if RUN_FIRES:
        key = os.environ.get("FIRMS_MAP_KEY", "")
        if not key:
            try:
                from google.colab import userdata
                key = userdata.get("FIRMS_MAP_KEY") or ""
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("NASA FIRMS MAP_KEY: ").strip()
        os.environ["FIRMS_MAP_KEY"] = key
        run("-m", "src.features.exogenous_drivers", "--start", "2024-01-01", "--end", "2025-12-31", "--fires")
    else:
        run("-m", "src.features.exogenous_drivers", "--start", "2024-01-01", "--end", "2025-12-31")
    run("-m", "src.models.ablation_exogenous")

p = rp("reports/modeling/exogenous", "ablation.md")
if require_fresh(p, RUN_WEATHER_ABLATION):
    display(Markdown(Path(p).read_text()))
    abl = pd.read_csv(rp("reports/modeling/exogenous", "ablation.csv"))
    v = abl[(abl["split"] == "val") & (abl["status"] == "ok")]
    pivot = v.pivot_table(index="group", columns=["panel", "family"], values="mae")
    pivot.plot(kind="bar", figsize=(10, 3.5), title="Exogenous ablation (validation MAE, lower is better)")
    plt.ylabel("MAE"); plt.tight_layout(); plt.show()
    print("not run:", sorted(set(abl.loc[abl["status"] == "not_run", "group"])) or "nothing")

## 7 — T2.3 · Raw PM2.5 target

Raw PM2.5 has real hourly dynamics, so skill over persistence (`pm25_now`) is visible, unlike for the
24 h-averaged AQI. For each horizon four HistGradientBoosting variants are fitted (target form
`level` / `residual` × calendar features on/off), the best on **validation** MAE is kept, and test is
scored once. This is the slowest section (12 fits).

In [ ]:
if RUN_PM25:
    run("-m", "src.models.train_pm25_forecast")
p = rp("reports/modeling/pm25", "pm25_forecast_metrics.csv")
if require_fresh(p, RUN_PM25):
    pm = pd.read_csv(p)
    display(Markdown(Path(rp("reports/modeling/pm25", "pm25_forecast_metrics.md")).read_text()))
    display(pd.read_csv(rp("reports/modeling/pm25", "variant_selection.csv")))
    t = pm[pm["split"] == "test"].set_index("horizon")
    y = t["skill_mae"]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(y.index, y, color=["#2a9d8f" if v > 0 else "#e76f51" for v in y],
           yerr=[y - t["skill_mae_ci_lo"], t["skill_mae_ci_hi"] - y], capsize=4)
    ax.axhline(0, color="k", lw=0.8); ax.set_title("PM2.5 test MAE skill vs persistence (95% CI)")
    plt.tight_layout(); plt.show()

## 8 — T2.5 · Probabilistic forecasts

Quantile models (α = 0.1 / 0.5 / 0.9, iteration counts chosen on validation pinball loss), a
finite-sample split-conformal pad on the 80 % interval, a persistence-band baseline for comparison, and a
classifier for `P(AQI ≥ 401)` scored with the Brier score against a persistence-based probability and the
constant-rate probability. The reliability diagram is drawn from test predictions.

In [ ]:
from IPython.display import Image
if RUN_QUANTILE:
    run("-m", "src.models.train_quantile_forecast")
d = out("reports/modeling/probabilistic")
if require_fresh(d / "quantile_metrics.csv", RUN_QUANTILE):
    display(Markdown((d / "quantile_metrics.md").read_text()))
    print(json.dumps(json.load(open(d / "probabilistic_metrics.json")), indent=2)[:6000])
    display(Image(str(d / "reliability_severe.png")))

## 9 — Headline table (single source of truth)

In [ ]:
ensure_frozen()  # the headline table reads the frozen predictions
if REBUILD_HEADLINE:
    run("src/evaluation/build_metrics_table.py")
p = "reports/metrics/headline_metrics.md"
if require_fresh(p, REBUILD_HEADLINE):
    display(Markdown(Path(p).read_text()))

## 10 — Wrap-up: which outputs are fresh, and download

The table lists every expected output and whether this run wrote it. Anything `STALE`/`MISSING` was not
produced in this run (section switched off or a script failed), so do not quote it.

In [ ]:
ensure_frozen()
import shutil
EXPECTED = {
    "T2.1 validation":   "reports/features/aqi_recompute_validation.md",
    "T2.1 extended 24h": "reports/modeling/24h/final/extended_training_metrics.json",
    "T2.2 rolling":      rp("reports/modeling/24h", "rolling_origin_backtest.csv"),
    "T2.4 ablation":     rp("reports/modeling/exogenous", "ablation.csv"),
    "T2.3 pm25":         rp("reports/modeling/pm25", "pm25_forecast_metrics.csv"),
    "T2.3 variants":     rp("reports/modeling/pm25", "variant_selection.csv"),
    "T2.5 quantiles":    rp("reports/modeling/probabilistic", "quantile_metrics.csv"),
    "T2.5 probabilistic": rp("reports/modeling/probabilistic", "probabilistic_metrics.json"),
    "T2.5 reliability":  rp("reports/modeling/probabilistic", "reliability_severe.png"),
    "headline":          "reports/metrics/headline_metrics.csv",
}
status = pd.DataFrame({
    "output": list(EXPECTED),
    "path": list(EXPECTED.values()),
    "state": ["FRESH" if fresh(p) else ("STALE" if Path(p).exists() else "MISSING") for p in EXPECTED.values()],
})
display(status)

import tempfile
out_dir = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
archive = shutil.make_archive(os.path.join(out_dir, "tier2_reports"), "zip", "reports")
print("archive:", archive, " (reports are also saved in the Drive project folder)")
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print("download skipped:", exc)